# Rina Park — Colab bootstrap

1. GPU 런타임 확인
2. Drive mount + Secrets (`HF_TOKEN`, `GIT_TOKEN`)
3. git clone → `requirements-colab.txt` → symlink → HF `--tier sdxl`
4. (선택) smoke `generate_ig_quality.py`

자세한 설명: `rina_park/ops/COLAB_SETUP.md`

In [ ]:
# 0) GPU check
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'Runtime → Change runtime type → GPU'

In [ ]:
# 1) Drive + secrets
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

def _secret(name: str) -> str:
    try:
        return userdata.get(name) or ''
    except Exception:
        return ''

hf = _secret('HF_TOKEN')
git_token = _secret('GIT_TOKEN')
if hf:
    os.environ['HF_TOKEN'] = hf
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf
print('HF_TOKEN:', 'yes' if hf else 'no')
print('GIT_TOKEN:', 'yes' if git_token else 'no')

In [ ]:
# 2) Clone (skip if already present)
from pathlib import Path
import shutil

REPO = Path('/content/ai_influencer')
url = 'https://github.com/pkang0831/ai_advertisement_brandings.git'
if git_token:
    url = f'https://{git_token}@github.com/pkang0831/ai_advertisement_brandings.git'

if REPO.exists() and not (REPO / 'rina_park').is_dir():
    shutil.rmtree(REPO)
if not (REPO / 'rina_park').is_dir():
    !git clone --depth 1 {url} {REPO}
else:
    print('already cloned', REPO)

%cd /content/ai_influencer
!git rev-parse --short HEAD

In [ ]:
# 3) Pip pins
!pip -q uninstall -y diffusers transformers accelerate huggingface_hub peft 2>/dev/null || true
!pip -q install --no-cache-dir -r requirements-colab.txt
print('pip ok')

In [ ]:
# 4) Symlink Drive ↔ rina_park
!python rina_park/scripts/colab_bootstrap.py
import os, sys
sys.path[:0] = ['/content/ai_influencer', '/content/ai_influencer/rina_park']
os.environ['PYTHONPATH'] = '/content/ai_influencer:/content/ai_influencer/rina_park'

In [ ]:
# 5) HF download — start with sdxl; change to wan / qwen_cuda / all later
TIER = 'sdxl'  # 'sdxl' | 'wan' | 'qwen_cuda' | 'all'
!python rina_park/scripts/colab_download_hf_models.py --tier {TIER}

In [ ]:
# 6) Smoke (needs RealVis + preferably character LoRA from Mac rclone)
from pathlib import Path
rv = Path('/content/ai_influencer/rina_park/models/checkpoints/RealVisXL_V5.0_fp16.safetensors')
lora = Path('/content/ai_influencer/rina_park/models/loras/rina_park_person_sdxl_lora.safetensors')
print('RealVis:', rv.exists(), rv)
print('character LoRA:', lora.exists(), lora)
if rv.exists():
    !PYTHONPATH=/content/ai_influencer:/content/ai_influencer/rina_park python rina_park/scripts/generate_ig_quality.py
else:
    print('Skip smoke — download sdxl tier first')